In [ ]:
import logging

import joblib
import numpy as np
import pandas as pd

from beir import LoggingHandler
from beir.retrieval import models
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch

from stemmer import Stemmer, tokenize

In [2]:
logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

In [ ]:
## EVAL PARAMS
MODEL_NAME = "../models/ams-bag_of_words" # "multi-qa-mpnet-base-dot-v1"
STEMMING_AT_EVAL = False

VOCAB_PATH = "../data/sundabaru1-vocab.txt"
BEIR_DATA_PATH = "../data/beir"
BEIR_QRELS_PATH = "../data/beir/qrels.tsv"

## Data Loading

In [ ]:
class AMSTokenizer:
    def __init__(self, stm: Stemmer):
        self.stemmer = stm

    def __call__(self, doc):
        return [self.stemmer.stem_ams(word) for word in tokenize(doc)]

In [ ]:
stemmer = Stemmer(VOCAB_PATH)
amstokenizer = AMSTokenizer(stemmer)

In [ ]:
def stem_sentence(text: str) -> str:
    return " ".join([stemmer.stem_ams(t) for t in tokenize(text)])

def stem_corpus(item: dict[str, str]):
    return {k: stem_sentence(v) for k, v in item.items()}

In [ ]:
corpus, queries, qrels = GenericDataLoader(data_folder=BEIR_DATA_PATH, qrels_file=BEIR_QRELS_PATH).load_custom()
if STEMMING_AT_EVAL:
    queries = stem_corpus(queries)
    corpus = {k: stem_corpus(v) for k, v in corpus.items()}

## Evaluate Model

### Create Evaluator for BoW & TF-IDF

In [ ]:
# https://github.com/beir-cellar/beir/wiki/Evaluate-your-custom-model
class BaselineModel:
    def __init__(self, model_name: str, vocab_path: str, **kwargs):
        self.model = joblib.load(model_name)
        if "stem" in model_name:
            self.model.tokenizer = AMSTokenizer(vocab_path)
    
    # Write your own encoding query function (Returns: Query embeddings as numpy array)
    def encode_queries(self, queries: list[str], batch_size: int, **kwargs) -> np.ndarray:
        return self.model.transform(queries).todense().astype(float)
    
    # Write your own encoding corpus function (Returns: Document embeddings as numpy array)  
    def encode_corpus(self, corpus: list[dict[str, str]], batch_size: int, **kwargs) -> np.ndarray:
        extracted_corpus = [row["title"] + " " + row["text"] for row in corpus]
        return self.model.transform(extracted_corpus).todense().astype(float)

dres_model = DenseRetrievalExactSearch(BaselineModel(MODEL_NAME, VOCAB_PATH), batch_size=16)
dres_model

### Create Evaluator for Sentence Transformers

In [ ]:
model = DenseRetrievalExactSearch(models.SentenceBERT(MODEL_NAME), batch_size=16)
model

2025-03-28 16:17:01 - Use pytorch device_name: cuda
2025-03-28 16:17:01 - Load pretrained SentenceTransformer: multi-qa-mpnet-base-dot-v1


## Evaluate

In [5]:
retriever = EvaluateRetrieval(model, score_function="dot")  # dot or cos_sim
results = retriever.retrieve(corpus, queries)

2025-03-28 16:17:30 - Encoding Queries...


Batches: 100%|██████████| 469/469 [00:08<00:00, 53.40it/s]


2025-03-28 16:17:39 - Sorting Corpus by document length (Longest first)...
2025-03-28 16:17:39 - Encoding Corpus in batches... Warning: This might take a while!
2025-03-28 16:17:39 - Scoring Function: Dot Product (dot)
2025-03-28 16:17:39 - Encoding Batch 1/1...


Batches: 100%|██████████| 94/94 [00:20<00:00,  4.51it/s]


In [ ]:
#### Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K  where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever.evaluate(qrels, results, retriever.k_values)
metrics = [
    {
        "model": MODEL_NAME, 
        "stemming": STEMMING_AT_EVAL,
        "metric": k.split("@")[0], 
        "k": k.split("@")[1], 
        "value": v
    } for col in [ndcg, _map, recall, precision] for k, v in col.items()
]

2025-03-28 16:18:21 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-03-28 16:18:24 - 

2025-03-28 16:18:24 - NDCG@1: 0.1236
2025-03-28 16:18:24 - NDCG@3: 0.1653
2025-03-28 16:18:24 - NDCG@5: 0.1778
2025-03-28 16:18:24 - NDCG@10: 0.1933
2025-03-28 16:18:24 - NDCG@100: 0.2376
2025-03-28 16:18:24 - NDCG@1000: 0.2876
2025-03-28 16:18:24 - 

2025-03-28 16:18:24 - MAP@1: 0.1236
2025-03-28 16:18:24 - MAP@3: 0.1549
2025-03-28 16:18:24 - MAP@5: 0.1618
2025-03-28 16:18:24 - MAP@10: 0.1682
2025-03-28 16:18:24 - MAP@100: 0.1761
2025-03-28 16:18:24 - MAP@1000: 0.1776
2025-03-28 16:18:24 - 

2025-03-28 16:18:24 - Recall@1: 0.1236
2025-03-28 16:18:24 - Recall@3: 0.1954
2025-03-28 16:18:24 - Recall@5: 0.2256
2025-03-28 16:18:24 - Recall@10: 0.2738
2025-03-28 16:18:24 - Recall@100: 0.4954
2025-03-28 16:18:24 - Recall@1000: 0.9146
2025-03-28 16:18:24 - 

2025-03-28 16:18:24 - P@1: 0.1236
2025-03-28 16:18:24

In [ ]:
df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv(f"eval-{MODEL_NAME}-{STEMMING_AT_EVAL}.csv", index=None)

df_metrics

,metric,k,value
0,NDCG,1,0.12362
1,NDCG,3,0.16533
2,NDCG,5,0.17777
3,NDCG,10,0.19328
4,NDCG,100,0.23762
5,NDCG,1000,0.28762
6,MAP,1,0.12362
7,MAP,3,0.15494
8,MAP,5,0.16185
9,MAP,10,0.16821
